# Open Design on Jute — M2 Deck Mode Design Spec

**Date:** 2026-06-01 · **Status:** Draft for review · **Authors:** brain (Claude) + Kevin
**Depends on:** host-shell (M1, merged) · asset-library (M3.5, merged)

> Specialize `kind: deck`. Decks are the one OD artifact type where Jute already has
> deep native infrastructure (cell↔slide, present mode, speaker notes, 12 layouts) —
> *and* where OD ships its richest assets (guizang-ppt + 51 `html-ppt-*` themes). M2
> reconciles the two with a **two-track** design.


## Context

M1 renders every artifact as a `text/html` cell output (the universal substrate).
M3.5 added the 148-system design library. **Decks are special twice over:**

- Jute has a *native, structured* deck mode — the notebook **is** the deck, one cell
  per slide, with present mode and speaker notes. No other artifact type has this.
- OD's deck assets are its most elaborate — a fixed-canvas HTML framework + 51 themed
  variants (magazine / WIRED / e-ink / launch styles, WebGL heroes).

These are two different rendering philosophies. M2's job is to use each where it wins.


## Verified finding A — Jute already has a native deck mode

Read from `src/ui/deck/*` + `src/bindings/JuteDeck*` + `src/agent/deck/*`:

- **12 slide layouts** (`JuteDeckLayout`): `auto · title · section · content · bullets ·
  code · output · code-output · two-col · image · blank` — each a React layout component.
- **Per-cell metadata** (`metadata.jute_deck`, `JuteDeckCellMetadata`): `layout`, `hidden`,
  `speaker_notes`, `theme_override`, `fragments` (bullet reveal), `background`.
- **Notebook metadata** (`JuteDeckNotebookMetadata`): `theme`, `aspect` (16:9), `title`, `author`.
- **`cellToSlide()`** maps a cell → slide, inferring layout from markdown (H1 → title, `##` →
  section, bullets → bullets) unless `jute_deck.layout` is explicit.
- **Present chrome + speaker notes** exist (`PresentChrome.tsx`, `SpeakerNotes.tsx`, `S` key overlay).
- **3 Tailwind themes only**: `minimal-light`, `minimal-dark`, `spur-brand` (class-based:
  frame / heading / body / muted / accent).
- **A deck-agent path already exists** (`dispatchDeckCommand` → delegates to a worker with the
  `mcp__notebook__*` allowlist) — overlaps with the `open-design` skill and must be reconciled.
- **`set_cell_metadata` already merges the `jute_deck` facet** — the brain can set per-slide
  layout/notes/theme today, no new tool needed.


## Verified finding B — OD decks are self-contained HTML artifacts

Read from `skills/guizang-ppt` + `skills/html-ppt-*` + `prompts/deck-framework.ts`:

- `mode: deck` skills emit **one self-contained HTML file**: a **1920×1080 fixed canvas**,
  `transform: scale()` scale-to-fit, capture-phase keyboard nav, `.slide.active` visibility,
  slide counter + print rules — all baked into `DECK_SKELETON_HTML` (injected verbatim so the
  model stops re-introducing scaling/focus bugs).
- **`guizang-ppt`** (magazine-web-ppt) is the flagship: WebGL fluid backgrounds, serif/sans
  pairing, section covers, big-number data slides.
- **51 `html-ppt-*` themes** + `simple-deck`, `replit-deck`, `weekly-update` are *taste variants*
  of that framework (editorial / brutalist / cyber-terminal / product-launch …).
- These are **not** structured layouts — they're rich opaque HTML. They do not map onto Jute's
  12 layout components or its 3 Tailwind themes.


## The core tension

```mermaid
flowchart TB
  deck([kind: deck]) --> q{which rendering model?}
  q -->|structured| nat["Jute NATIVE deck mode\ncell ↔ slide · 12 layouts · present\nspeaker notes · fragments · 3 themes"]
  q -->|opaque HTML| art["OD HTML-ARTIFACT deck\n1920×1080 fixed canvas · scale-to-fit\n51 themed variants · WebGL heroes"]
  nat --> natpro["✅ editable, reactive, native nav\n❌ limited themes, no rich templates"]
  art --> artpro["✅ 51 rich themes verbatim, pixel-fidelity\n❌ opaque blob, no cell↔slide / present"]
  classDef a fill:#e3f2fd,stroke:#1e88e5;
  classDef b fill:#fff8e1,stroke:#fb8c00;
  class nat,natpro a; class art,artpro b;
```

Forcing OD's 51 HTML themes into Jute's structured renderer is lossy; forcing every deck
through an opaque HTML blob throws away the native deck mode Jute already has. So: **two tracks.**


## Locked decisions — two-track, native-first

| # | Decision | Choice |
|---|----------|--------|
| 1 | One model or two? | **Two tracks**: native structured deck + HTML-artifact deck |
| 2 | Default track | **Native-first** — the brain defaults to Jute's native deck mode (it's editable, reactive, and already built); escalate to the artifact track only for "polished/branded presentation" briefs |
| 3 | Native theming | Port a **small set** of OD directions into Jute `THEMES` (Tailwind/CSS tokens); do NOT try to recreate 51 rich templates as structured themes |
| 4 | The 51 `html-ppt-*` | Land as the **artifact-track theme library**, indexed like M3.5's design systems (id → mode/scenario/preview), rendered via OD's `DECK_SKELETON_HTML` as a `text/html` cell |
| 5 | Per-slide control | Reuse the existing `jute_deck` facet via `set_cell_metadata` (no new tool) |
| 6 | Agent path | **Reconcile** `dispatchDeckCommand` with the `open-design` skill — one deck flow driven by the brain through `notebook_*`, not two |


## Native deck track — design

The notebook **is** the deck. The `open-design` deck flow, on `kind: deck`:

1. Set notebook `metadata.jute_deck` (`theme`, `aspect`, `title`, `author`).
2. Emit **one cell per slide** (markdown for prose slides, code for live/output slides).
3. Set each slide's `jute_deck` facet via `set_cell_metadata` — `layout`, `speaker_notes`,
   `fragments`, `background`, `theme_override`.
4. Let `cellToSlide()` + the 12 layout components render; present mode / speaker notes / `S`
   overlay already work.

```mermaid
sequenceDiagram
  participant A as Brain (open-design · kind=deck)
  participant N as Notebook (notebook_* + set_cell_metadata)
  A->>N: set notebook jute_deck {theme, aspect:"16:9", title}
  loop per slide
    A->>N: insert_cell(markdown/code, slide source)
    A->>N: set_cell_metadata(jute_deck:{layout, speaker_notes, fragments})
  end
  Note over N: cellToSlide → layout components → present mode
  A->>A: critique (deck-aware checklist) → revise slide cells
```

**Layout mapping** the skill teaches: `# H1`→`title`, `## H2`→`section`, bullets→`bullets`,
code→`code`/`code-output`, image→`image`, else `content`/`two-col`. Explicit `layout` overrides inference.


## Artifact deck track — design

For "make it a polished magazine/launch deck" briefs, render OD's framework as one
`text/html` cell (the M1 substrate), themed by one of the 51 `html-ppt-*`.

```mermaid
flowchart LR
  brief([polished deck brief]) --> pick["open_design_search(mode=deck)\n→ guizang-ppt / html-ppt-<taste>"]
  pick --> get["get theme package\n(SKELETON + template.html)"]
  get --> build["author slides into DECK_SKELETON_HTML\n(1920×1080, scale-to-fit baked in)"]
  build --> cell["text/html cell output\nsandboxed iframe (allow-scripts)"]
  classDef n fill:#fff8e1,stroke:#fb8c00;
  class pick,get,build,cell n;
```

The 51 themes + `guizang-ppt` are vendored + indexed exactly like M3.5's design systems
(brain-vendored, `index.json`: `{id, taste, scenario, preview, summary}`). `DECK_SKELETON_HTML`
ships once (the skill copies it verbatim — its whole point is to stop re-deriving scale-to-fit JS).


## Track-selection rule (what the brain decides)

| Brief signal | Track |
|---|---|
| "working deck", "outline", "I'll edit slides", data/charts, reactive | **Native** (default) |
| "magazine", "launch", "pitch for investors", "polished", named taste (WIRED/editorial/brutalist), WebGL/hero | **Artifact** (51 themes) |
| unsure | **Native** — it's editable; the user can escalate |

## Theme bridge

| Source | Count | Lands as |
|---|---|---|
| Jute built-in | 3 | `minimal-light/dark`, `spur-brand` (keep) |
| OD 5 directions → ported | 5 | new Jute `THEMES` entries (Tailwind/CSS tokens) — native track |
| OD `html-ppt-*` + guizang | 52 | artifact-track theme library (indexed, HTML, verbatim) |


## UI/UX mockup — native deck present view (rendered as a text/html cell output)

Illustrative present-mode slide: a `title`-layout cell, slide counter outside the scaled
stage, and the `S`-overlay speaker-notes hint — all native Jute deck mode.


In [1]:
# Native deck present-view mockup (cell -> slide, layout=title)

## Milestones within M2

- **M2a — Native deck track (core).** Wire the `open-design` deck flow to Jute's native deck
  mode (notebook + per-cell `jute_deck` via `set_cell_metadata`, layout mapping, present).
  Reconcile `dispatchDeckCommand` into the one `open-design` deck flow. Deck-aware critique.
- **M2b — Theme port.** Port the 5 OD directions into Jute `THEMES` so native decks aren't
  limited to 3 looks.
- **M2c — Artifact deck track + theme library.** Vendor `guizang-ppt` + 51 `html-ppt-*` (brain-
  vendored; gitignored upstream) + index; ship `DECK_SKELETON_HTML`; wire the polished-deck path.


## Open decisions

1. **`dispatchDeckCommand` reconciliation** — fold it into the `open-design` skill, or keep it
   as a thin UI entry that invokes the same flow? (Lean: one flow; UI button just seeds the brief.)
2. **Theme model for the native track** — extend `THEMES` (Tailwind classes) only, or allow a
   per-deck CSS-token theme so ported OD palettes keep their type stacks? (Lean: CSS tokens.)
3. **`DECK_SKELETON_HTML` home** — bundle in the `open-design` skill refs, or in the asset
   library beside the deck themes? (Lean: asset library, beside the 51 themes.)
4. **Export** — native deck → PDF/PPTX vs artifact deck → print-to-PDF; unify or per-track? (M4-ish.)
5. **Vendor scope for M2c** — all 52 deck themes day one, or `guizang-ppt` + a curated ~8 first?

**Next:** on approval, turn **M2a (native deck track)** into the next `submit_plan` — it's the
highest-integration, lowest-new-infra slice (the deck mode already exists).


## Sources

- Jute deck mode: `src/ui/deck/{cellToSlide,themes,SlideFrame,PresentChrome,SpeakerNotes,layouts/*}`,
  `src/bindings/JuteDeck{CellMetadata,NotebookMetadata,Layout}.ts`, `src/agent/deck/{dispatch,prompts}.ts`
- OD deck: `skills/guizang-ppt`, `skills/html-ppt-*`, `packages/contracts/src/prompts/deck-framework.ts`
- Prior specs: host-shell (2026-05-31), asset-library (2026-06-01)
